# Cleaning and Integration

This notebook documents the cleaning, normalization, and join workflow for the crop-yield prediction challenge.

In [ ]:
# Save master tables
train_master.to_csv(PROCESSED_DIR / 'master_train.csv', index=False)
test_master.to_csv(PROCESSED_DIR / 'master_test.csv', index=False)
cleaning_log_df.to_csv(Path('../reports') / 'A_cleaning_log.csv', index=False)

print("Master tables saved:")
print(f"  {PROCESSED_DIR / 'master_train.csv'}")
print(f"  {PROCESSED_DIR / 'master_test.csv'}")

# Create data dictionary
data_dict = make_data_dictionary(train_master)
data_dict.to_csv(PROCESSED_DIR / 'data_dictionary_master.csv', index=False)
print(f"  {PROCESSED_DIR / 'data_dictionary_master.csv'}")
print(f"\nData dictionary preview:")
print(data_dict.head(15))

In [ ]:
# Add weather features
train_master = add_weather_features(train_clean, weather_clean)
test_master = add_weather_features(test_clean, weather_clean)

print(f"\nAfter weather join:")
print(f"  Train with weather: {train_master.shape}")
print(f"  Test with weather: {test_master.shape}")
print(f"  Feature columns: {[c for c in train_master.columns if 'season' in c or 'planting' in c]}")

In [ ]:
# Clean each raw dataset
train_clean, train_log = clean_plot_data(train_raw)
test_clean, test_log = clean_plot_data(test_raw)
weather_clean, weather_log = clean_weather_data(weather_raw)
price_clean, price_log = clean_market_price(price_raw)

print("\nAfter cleaning:")
print(f"  Train: {train_clean.shape}")
print(f"  Test: {test_clean.shape}")
print(f"  Weather: {weather_clean.shape}")
print(f"  Price: {price_clean.shape}")

# Build cleaning log
cleaning_log = build_cleaning_log(train_log, weather_log, price_log)
print(f"\nCleaning log entries: {len(cleaning_log)}")
cleaning_log_df = pd.DataFrame(cleaning_log)
print(cleaning_log_df.head(10))

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import pandas as pd
import numpy as np
from cleaning import clean_plot_data, clean_weather_data, clean_market_price, build_cleaning_log, standardize_region, standardize_crop
from features import add_weather_features, attach_price_analysis, build_feature_columns, make_data_dictionary

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

# Load raw data
train_raw = pd.read_csv(RAW_DIR / 'crop_yield_train.csv')
test_raw = pd.read_csv(RAW_DIR / 'crop_yield_leaderboard_test.csv')
weather_raw = pd.read_csv(RAW_DIR / 'regional_weather.csv')
price_raw = pd.read_csv(RAW_DIR / 'market_prices.csv')

print("Raw data shapes:")
print(f"  Train: {train_raw.shape}")
print(f"  Test: {test_raw.shape}")
print(f"  Weather: {weather_raw.shape}")
print(f"  Price: {price_raw.shape}")